# Vesuvius Challenge 2025 - TPU Training (Keras/JAX)

## Model: 6-stage ResEncUNet3D with scSE + Topology-Aware Losses

**Target Hardware:** Kaggle TPU v5e-8

**Data Format:** TIFF files (same as GPU version - no TFRecords needed)

### Architecture (Same as PyTorch version):
| Component | Specification |
|-----------|---------------|
| Model | 6-stage ResEncUNet3D |
| Features | [32, 64, 128, 256, 320, 320] |
| Blocks | [1, 3, 4, 6, 6, 6] |
| Attention | scSE (Spatial-Channel Squeeze-Excitation) |
| Normalization | GroupNorm |
| Deep Supervision | Yes (3 heads) |
| Patch Size | 192×192×192 |

### Loss Schedule:
- Epochs 0-299: Dice + BCE
- Epochs 300-599: Add SkeletonRecall
- Epochs 600-1200: Add soft-clDice

### Expected Data Structure:
```
/kaggle/input/3d-volume-training-data/
├── train.csv
├── train_images/
│   ├── volume_001.tif
│   ├── volume_002.tif
│   └── ...
└── train_labels/
    ├── volume_001.tif
    ├── volume_002.tif
    └── ...
```

In [ ]:
# =============================================================================
# CELL 1: SETUP & INSTALLATION
# =============================================================================

from IPython.display import clear_output

# Required for TPU training with JAX backend
!pip install tensorflow -qU

# Install latest keras (for TPU compatibility)
!pip install keras --upgrade -q

# Install tifffile for loading TIFF volumes
!pip install tifffile imagecodecs -q

clear_output()
print("Dependencies installed")

In [ ]:
# =============================================================================
# CELL 2: IMPORTS & TPU CONFIGURATION
# =============================================================================

import os
import warnings

# Set JAX backend BEFORE importing keras
os.environ["KERAS_BACKEND"] = "jax"
warnings.filterwarnings('ignore')

import gc
import glob
import json
import random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import tifffile

# Keras 3 with JAX backend
import keras
from keras import ops
from keras import layers
from keras import Model
from keras.optimizers import AdamW
from keras.optimizers.schedules import CosineDecay

# TensorFlow for data pipeline only
import tensorflow as tf

# Disable flash attention for TPU compatibility
keras.config.disable_flash_attention()

# Set random seed for reproducibility
keras.utils.set_random_seed(42)

# =============================================================================
# TPU DISTRIBUTION SETUP
# =============================================================================

devices = keras.distribution.list_devices()
data_parallel = keras.distribution.DataParallel(devices=devices)
keras.distribution.set_distribution(data_parallel)
total_device = len(devices)

print(f"Keras version: {keras.version()}")
print(f"Backend: {keras.config.backend()}")
print(f"Detected devices: {devices}")
print(f"Total devices: {total_device}")

In [ ]:
# =============================================================================
# CELL 3: CONFIGURATION
# =============================================================================

@dataclass
class Config:
    # Data paths (UPDATE THESE FOR YOUR KAGGLE DATASET)
    DATA_ROOT: str = "/kaggle/input/3d-volume-training-data"
    CHECKPOINT_DIR: str = "/kaggle/working/checkpoints"
    
    # Model architecture (MUST MATCH PyTorch version)
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = None
    N_BLOCKS: List[int] = None
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    NUM_CLASSES: int = 1  # Binary segmentation
    
    # Training settings
    EPOCHS: int = 1200
    BATCH_SIZE_PER_DEVICE: int = 1
    PATCHES_PER_VOLUME: int = 8  # Number of patches to extract per volume per epoch
    
    # Learning rate
    INITIAL_LR: float = 1e-6
    PEAK_LR: float = 3e-4
    WARMUP_RATIO: float = 0.05
    WEIGHT_DECAY: float = 1e-5
    
    # Loss weights
    DICE_WEIGHT: float = 0.3
    BCE_WEIGHT: float = 0.2
    SKELETON_WEIGHT: float = 0.25
    CLDICE_WEIGHT: float = 0.25
    
    # Loss scheduling (epochs)
    SKELETON_START_EPOCH: int = 300
    SKELETON_WARMUP_EPOCHS: int = 300
    CLDICE_START_EPOCH: int = 600
    CLDICE_WARMUP_EPOCHS: int = 300
    
    # Validation
    VALIDATE_EVERY: int = 5
    VAL_OVERLAP: float = 0.5
    
    # Data loading
    DATA_FRACTION: float = 1.0  # Use fraction of data for quick testing
    PRELOAD_DATA: bool = True   # Preload all data to RAM
    
    # Random seed
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]
        
        # Create checkpoint directory
        os.makedirs(self.CHECKPOINT_DIR, exist_ok=True)
        
        # Calculate global batch size
        self.BATCH_SIZE = self.BATCH_SIZE_PER_DEVICE * total_device
        
        # Scaled peak LR based on batch size
        self.PEAK_LR = min(3e-4, 1e-4 * (self.BATCH_SIZE / 2))


cfg = Config()
cfg.__post_init__()

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Data root: {cfg.DATA_ROOT}")
print(f"Patch size: {cfg.PATCH_SIZE}")
print(f"Features: {cfg.FEATURES}")
print(f"Blocks: {cfg.N_BLOCKS}")
print(f"Batch size per device: {cfg.BATCH_SIZE_PER_DEVICE}")
print(f"Global batch size: {cfg.BATCH_SIZE}")
print(f"Patches per volume: {cfg.PATCHES_PER_VOLUME}")
print(f"Epochs: {cfg.EPOCHS}")
print(f"Peak LR: {cfg.PEAK_LR}")
print("="*60)

In [ ]:
# =============================================================================
# CELL 4: MODEL DEFINITION (6-stage ResEncUNet3D + scSE)
# =============================================================================

class ConvBlock(layers.Layer):
    """3D Convolution block with GroupNorm and LeakyReLU."""
    
    def __init__(self, out_channels, num_groups=8, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels
        
        # Ensure groups divide channels evenly
        num_groups = min(num_groups, out_channels)
        while out_channels % num_groups != 0:
            num_groups -= 1
        self.num_groups = num_groups
    
    def build(self, input_shape):
        self.conv = layers.Conv3D(
            self.out_channels, 
            kernel_size=3, 
            padding='same', 
            use_bias=False
        )
        self.norm = layers.GroupNormalization(groups=self.num_groups)
        self.act = layers.LeakyReLU(negative_slope=0.01)
        super().build(input_shape)
    
    def call(self, x):
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x


class ResBlock(layers.Layer):
    """Residual block with N convolutions."""
    
    def __init__(self, channels, n_convs=2, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.n_convs = n_convs
    
    def build(self, input_shape):
        self.blocks = [ConvBlock(self.channels) for _ in range(self.n_convs)]
        super().build(input_shape)
    
    def call(self, x):
        residual = x
        for block in self.blocks:
            x = block(x)
        return x + residual


class scSEBlock(layers.Layer):
    """Concurrent Spatial and Channel Squeeze-and-Excitation (scSE) for 3D."""
    
    def __init__(self, channels, reduction=2, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.reduction = reduction
    
    def build(self, input_shape):
        # Channel SE
        self.cse_pool = layers.GlobalAveragePooling3D()
        self.cse_fc1 = layers.Dense(self.channels // self.reduction, activation='relu')
        self.cse_fc2 = layers.Dense(self.channels, activation='sigmoid')
        
        # Spatial SE
        self.sse_conv = layers.Conv3D(1, kernel_size=1, activation='sigmoid')
        super().build(input_shape)
    
    def call(self, x):
        # Channel SE path
        cse = self.cse_pool(x)  # (batch, channels)
        cse = self.cse_fc1(cse)
        cse = self.cse_fc2(cse)  # (batch, channels)
        cse = ops.reshape(cse, (-1, 1, 1, 1, self.channels))  # (batch, 1, 1, 1, channels)
        
        # Spatial SE path
        sse = self.sse_conv(x)  # (batch, D, H, W, 1)
        
        # Combine: channel-weighted + spatial-weighted
        return x * cse + x * sse


class ResEncUNet3D(Model):
    """
    6-stage Residual Encoder U-Net with scSE attention.
    
    Keras/JAX implementation matching the PyTorch version exactly.
    Uses channel-last format: (batch, D, H, W, C)
    """
    
    def __init__(
        self,
        in_channels: int = 1,
        out_channels: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
        **kwargs
    ):
        super().__init__(**kwargs)
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.n_blocks_list = n_blocks
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)
        self.out_channels = out_channels
        
        # Encoder stages
        self.encoders = []
        self.attentions = []
        self.pools = []
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            # Encoder: initial conv + residual blocks
            encoder_layers = [ConvBlock(feat)]
            encoder_layers.extend([ResBlock(feat, n_convs=2) for _ in range(nb)])
            self.encoders.append(encoder_layers)
            
            # Attention
            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(None)
            
            # Downsampling (strided conv)
            if i < len(features) - 1:
                self.pools.append(
                    layers.Conv3D(feat, kernel_size=2, strides=2, padding='same')
                )
        
        # Decoder stages
        self.ups = []
        self.dec_convs = []
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(
                layers.Conv3DTranspose(out_feat, kernel_size=2, strides=2, padding='same')
            )
            self.dec_convs.append(ConvBlock(out_feat))
        
        # Final output
        self.final_conv = layers.Conv3D(out_channels, kernel_size=1)
        
        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = []
            n_ds_outputs = min(3, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(layers.Conv3D(out_channels, kernel_size=1))
    
    def call(self, x, training=False):
        enc_features = []
        
        # Encoder path
        for i, (encoder_layers, attention) in enumerate(zip(self.encoders, self.attentions)):
            for layer in encoder_layers:
                x = layer(x)
            
            if attention is not None:
                x = attention(x)
            
            enc_features.append(x)
            
            if i < len(self.pools):
                x = self.pools[i](x)
        
        # Reverse for decoder
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        ds_outputs = []
        
        # Decoder path
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            
            # Handle size mismatch
            if x.shape[1:4] != skip.shape[1:4]:
                target_shape = skip.shape[1:4]
                x = ops.image.resize(
                    x, 
                    size=target_shape,
                    interpolation='trilinear'
                )
            
            # Concatenate skip connection
            x = ops.concatenate([x, skip], axis=-1)
            x = dec(x)
            
            # Deep supervision outputs
            if self.use_deep_supervision and training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final_conv(x)
        
        if self.use_deep_supervision and training:
            return {'output': out, 'deep': ds_outputs}
        return out


def build_model(cfg):
    """Build and return the model."""
    model = ResEncUNet3D(
        in_channels=1,
        out_channels=cfg.NUM_CLASSES,
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    )
    
    # Build model by calling it once
    dummy_input = ops.zeros((1,) + cfg.PATCH_SIZE + (1,))
    _ = model(dummy_input, training=True)
    
    return model


# Build and check model
model = build_model(cfg)
print(f"Model parameters: {model.count_params():,}")
print("Model built successfully!")

In [ ]:
# =============================================================================
# CELL 5: LOSS FUNCTIONS (Keras Loss compatible with model.fit())
# =============================================================================

def dice_loss(y_true, y_pred, smooth=1e-5):
    """
    Dice loss.
    
    Args:
        y_true: Ground truth (batch, D, H, W, 1)
        y_pred: Predictions - logits (batch, D, H, W, 1)
        smooth: Smoothing factor
    """
    y_pred = ops.sigmoid(y_pred)
    
    intersection = ops.sum(y_pred * y_true)
    union = ops.sum(y_pred) + ops.sum(y_true)
    
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice


def bce_loss(y_true, y_pred):
    """
    Binary Cross-Entropy loss.
    """
    return ops.mean(
        keras.losses.binary_crossentropy(y_true, y_pred, from_logits=True)
    )


def soft_skeletonize(x, num_iter=10):
    """
    Soft skeletonization using iterative min-max pooling.
    From clDice paper (CVPR 2021).
    
    Input shape: (batch, D, H, W, C)
    """
    for _ in range(num_iter):
        # Min pool (erosion-like)
        min_pool = -ops.max_pool(
            -x, 
            pool_size=(3, 3, 3), 
            strides=(1, 1, 1), 
            padding='same'
        )
        # Max pool of min pool
        max_min_pool = ops.max_pool(
            min_pool, 
            pool_size=(3, 3, 3), 
            strides=(1, 1, 1), 
            padding='same'
        )
        x = ops.relu(x - max_min_pool)
    return x


def soft_cldice_loss(y_true, y_pred, smooth=1e-5, num_iter=10):
    """
    Soft clDice (centerline Dice) loss.
    Preserves topology by focusing on skeleton connectivity.
    """
    y_pred = ops.sigmoid(y_pred)
    
    skel_pred = soft_skeletonize(y_pred, num_iter)
    skel_true = soft_skeletonize(y_true, num_iter)
    
    # Topology precision: skeleton(pred) overlaps with target
    tprec = (ops.sum(skel_pred * y_true) + smooth) / (ops.sum(skel_pred) + smooth)
    
    # Topology sensitivity: skeleton(target) overlaps with pred
    tsens = (ops.sum(skel_true * y_pred) + smooth) / (ops.sum(skel_true) + smooth)
    
    cldice = 2.0 * tprec * tsens / (tprec + tsens + smooth)
    return 1.0 - cldice


def skeleton_recall_loss(y_true, y_pred, smooth=1e-5, tube_radius=2):
    """
    Skeleton Recall Loss (ECCV 2024).
    Focuses on recalling the skeleton of the target.
    """
    y_pred_sigmoid = ops.sigmoid(y_pred)
    
    # Extract target skeleton
    target_skel = soft_skeletonize(y_true, num_iter=10)
    
    # Dilate skeleton to create tube
    if tube_radius > 0:
        kernel_size = 2 * tube_radius + 1
        target_tube = ops.max_pool(
            target_skel,
            pool_size=(kernel_size, kernel_size, kernel_size),
            strides=(1, 1, 1),
            padding='same'
        )
    else:
        target_tube = target_skel
    
    # Recall: how much of the skeleton tube is covered by prediction
    recall = ops.sum(y_pred_sigmoid * target_tube) / (ops.sum(target_tube) + smooth)
    return 1.0 - recall


class CombinedLoss(keras.losses.Loss):
    """
    Combined loss with progressive scheduling for topology losses.
    Inherits from keras.losses.Loss for compatibility with model.fit().
    
    Loss schedule:
    - Epochs 0-skeleton_start: Dice + BCE only
    - Epochs skeleton_start to cldice_start: Add SkeletonRecall
    - Epochs cldice_start onwards: Add soft-clDice
    """
    
    def __init__(
        self,
        dice_weight: float = 0.3,
        bce_weight: float = 0.2,
        skeleton_weight: float = 0.25,
        cldice_weight: float = 0.25,
        skeleton_start: int = 300,
        skeleton_warmup: int = 300,
        cldice_start: int = 600,
        cldice_warmup: int = 300,
        **kwargs
    ):
        super().__init__(name='combined_loss', **kwargs)
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.skeleton_weight = skeleton_weight
        self.cldice_weight = cldice_weight
        
        self.skeleton_start = skeleton_start
        self.skeleton_warmup = skeleton_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup
        
        # Current epoch (will be updated during training via callback)
        self.current_epoch = 0
    
    def _get_scale(self, epoch, start, warmup):
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def call(self, y_true, y_pred):
        """
        Compute combined loss.
        
        Args:
            y_true: Ground truth
            y_pred: Model prediction (logits)
        """
        epoch = self.current_epoch
        
        skel_scale = self._get_scale(epoch, self.skeleton_start, self.skeleton_warmup)
        cldice_scale = self._get_scale(epoch, self.cldice_start, self.cldice_warmup)
        
        # Base losses
        dice = dice_loss(y_true, y_pred)
        bce = bce_loss(y_true, y_pred)
        
        # Start with base losses
        total = self.dice_weight * dice + self.bce_weight * bce
        
        # Add topology losses with scheduling
        if skel_scale > 0:
            skel = skeleton_recall_loss(y_true, y_pred)
            total = total + self.skeleton_weight * skel_scale * skel
        
        if cldice_scale > 0:
            cldice = soft_cldice_loss(y_true, y_pred)
            total = total + self.cldice_weight * cldice_scale * cldice
        
        return total
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'dice_weight': self.dice_weight,
            'bce_weight': self.bce_weight,
            'skeleton_weight': self.skeleton_weight,
            'cldice_weight': self.cldice_weight,
            'skeleton_start': self.skeleton_start,
            'skeleton_warmup': self.skeleton_warmup,
            'cldice_start': self.cldice_start,
            'cldice_warmup': self.cldice_warmup,
        })
        return config


print("Loss functions defined (Keras Loss compatible)")
print(f"Loss schedule:")
print(f"  Epochs 0-{cfg.SKELETON_START_EPOCH}: Dice + BCE only")
print(f"  Epochs {cfg.SKELETON_START_EPOCH}-{cfg.CLDICE_START_EPOCH}: Add SkeletonRecall")
print(f"  Epochs {cfg.CLDICE_START_EPOCH}+: Add soft-clDice")

In [ ]:
# =============================================================================
# CELL 6: DATA PIPELINE (TIFF Files - Compatible with model.fit())
# =============================================================================

def load_volume_tiff(path: str) -> np.ndarray:
    """Load 3D TIF volume."""
    return tifffile.imread(str(path))


class VesuviusDataset:
    """
    Dataset for Vesuvius 3D volumes from TIFF files.
    
    Preloads all data to RAM for efficient TPU training.
    Same approach as the GPU PyTorch version.
    """
    
    def __init__(
        self,
        data_root: str,
        patch_size: Tuple[int, int, int] = (192, 192, 192),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 8,
        preload: bool = True,
    ):
        self.data_root = Path(data_root)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        
        # Find data files
        train_csv = self.data_root / "train.csv"
        images_dir = self.data_root / "train_images"
        labels_dir = self.data_root / "train_labels"
        
        # Load CSV and filter valid IDs
        df = pd.read_csv(train_csv)
        valid_ids = []
        for idx in df['id'].values:
            img_path = images_dir / f"{idx}.tif"
            lbl_path = labels_dir / f"{idx}.tif"
            if img_path.exists() and lbl_path.exists():
                valid_ids.append(idx)
        
        # Apply data fraction
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        
        # Storage for preloaded data
        self.volumes: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
        self.fg_coords: Dict[str, Optional[np.ndarray]] = {}
        
        # Preload all data to RAM
        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes to RAM...")
            from tqdm.auto import tqdm
            
            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                img = load_volume_tiff(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume_tiff(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)
                
                # Normalize image
                img = (img - img.mean()) / (img.std() + 1e-8)
                
                self.volumes[vol_id] = (img, lbl)
                
                # Cache foreground coordinates for biased sampling
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None
            
            # Estimate memory usage
            sample_vol = next(iter(self.volumes.values()))
            vol_size_mb = (sample_vol[0].nbytes + sample_vol[1].nbytes) / 1e6
            total_gb = vol_size_mb * len(self.volumes) / 1000
            print(f"Loaded {len(self.volumes)} volumes ({total_gb:.1f} GB in RAM)")
        
        print(f"Dataset ready: {len(self)} samples ({'train' if is_train else 'val'})")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def get_sample(self, idx: int) -> Dict[str, np.ndarray]:
        """Get a single training sample."""
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self.volumes[vol_id]
        
        # Extract patch
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if volume is smaller than patch
        if d < pd or h < ph or w < pw:
            pad_d = max(0, pd - d)
            pad_h = max(0, ph - h)
            pad_w = max(0, pw - w)
            image = np.pad(image, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect')
            label = np.pad(label, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape
        
        # Foreground-centered sampling (60% of time during training)
        fg = self.fg_coords.get(vol_id)
        if self.is_train and random.random() < 0.6 and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg) - 1)]
            z = max(0, min(c[0] - pd // 2, d - pd))
            y = max(0, min(c[1] - ph // 2, h - ph))
            x = max(0, min(c[2] - pw // 2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()
        
        # Augmentation (training only)
        if self.is_train:
            # Random flips
            for ax in range(3):
                if random.random() > 0.5:
                    img_patch = np.flip(img_patch, ax)
                    lbl_patch = np.flip(lbl_patch, ax)
            
            # Random 90-degree rotation
            k = random.randint(0, 3)
            if k:
                img_patch = np.rot90(img_patch, k, (1, 2))
                lbl_patch = np.rot90(lbl_patch, k, (1, 2))
            
            # Make contiguous
            img_patch = np.ascontiguousarray(img_patch)
            lbl_patch = np.ascontiguousarray(lbl_patch)
            
            # Intensity augmentation
            if random.random() > 0.5:
                img_patch = img_patch * random.uniform(0.8, 1.2)
            if random.random() > 0.5:
                img_patch = img_patch + random.uniform(-0.1, 0.1)
        
        # Create mask (label==1 is foreground, label==2 is ignore -> treat as background)
        fg_mask = (lbl_patch == 1).astype(np.float32)
        
        # Add channel dimension: (D, H, W) -> (D, H, W, 1)
        return {
            'image': img_patch[..., np.newaxis].astype(np.float32),
            'label': fg_mask[..., np.newaxis].astype(np.float32),
        }


def create_tf_dataset_for_fit(vesuvius_dataset: VesuviusDataset, batch_size: int, shuffle: bool = True):
    """
    Create tf.data.Dataset compatible with model.fit().
    
    Returns (image, label) tuples instead of dict for Keras compatibility.
    Uses tf.data for efficient batching and prefetching on TPU.
    """
    patch_size = vesuvius_dataset.patch_size
    
    def generator():
        indices = list(range(len(vesuvius_dataset)))
        if shuffle:
            random.shuffle(indices)
        for idx in indices:
            sample = vesuvius_dataset.get_sample(idx)
            # Return (x, y) tuple for model.fit()
            yield sample['image'], sample['label']
    
    # Create dataset from generator
    output_signature = (
        tf.TensorSpec(shape=patch_size + (1,), dtype=tf.float32),
        tf.TensorSpec(shape=patch_size + (1,), dtype=tf.float32),
    )
    
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=output_signature
    )
    
    dataset = dataset.batch(batch_size, drop_remainder=shuffle)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset


print("TIFF-based data pipeline defined (model.fit() compatible)")
print(f"Expected data structure:")
print(f"  {cfg.DATA_ROOT}/")
print(f"    train.csv")
print(f"    train_images/*.tif")
print(f"    train_labels/*.tif")

In [ ]:
# =============================================================================
# CELL 7: LOAD DATA & TRAINING SETUP
# =============================================================================

# Load training data
print("[1/2] Loading training data...")
train_dataset = VesuviusDataset(
    data_root=cfg.DATA_ROOT,
    patch_size=cfg.PATCH_SIZE,
    is_train=True,
    data_fraction=cfg.DATA_FRACTION,
    patches_per_volume=cfg.PATCHES_PER_VOLUME,
    preload=cfg.PRELOAD_DATA,
)

# Load validation data (smaller fraction)
print("\n[2/2] Loading validation data...")
val_dataset = VesuviusDataset(
    data_root=cfg.DATA_ROOT,
    patch_size=cfg.PATCH_SIZE,
    is_train=False,
    data_fraction=0.15,  # Use 15% for validation
    patches_per_volume=1,
    preload=cfg.PRELOAD_DATA,
)

# Calculate steps
num_samples = len(train_dataset)
steps_per_epoch = num_samples // cfg.BATCH_SIZE
total_steps = steps_per_epoch * cfg.EPOCHS
warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
decay_steps = max(1, total_steps - warmup_steps)

print(f"\nTraining samples: {num_samples}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

# Learning rate schedule
lr_schedule = CosineDecay(
    initial_learning_rate=cfg.INITIAL_LR,
    decay_steps=decay_steps,
    warmup_target=cfg.PEAK_LR,
    warmup_steps=warmup_steps,
    alpha=0.1,  # Final LR = alpha * peak_lr
)

# Optimizer
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=cfg.WEIGHT_DECAY,
)

print(f"\nOptimizer: AdamW")
print(f"Initial LR: {cfg.INITIAL_LR}")
print(f"Peak LR: {cfg.PEAK_LR}")
print(f"Weight decay: {cfg.WEIGHT_DECAY}")

In [ ]:
# =============================================================================
# CELL 8: METRICS
# =============================================================================

class DiceMetric(keras.metrics.Metric):
    """Dice coefficient metric with ignore mask support."""
    
    def __init__(self, threshold=0.5, smooth=1e-5, name='dice', **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.smooth = smooth
        self.intersection = self.add_weight(name='intersection', initializer='zeros')
        self.union = self.add_weight(name='union', initializer='zeros')
    
    def update_state(self, y_true, y_pred, ignore_mask=None, sample_weight=None):
        # Apply sigmoid if logits
        y_pred = ops.sigmoid(y_pred)
        y_pred = ops.cast(y_pred > self.threshold, 'float32')
        
        if ignore_mask is not None:
            valid_mask = 1.0 - ignore_mask
            y_pred = y_pred * valid_mask
            y_true = y_true * valid_mask
        
        intersection = ops.sum(y_pred * y_true)
        union = ops.sum(y_pred) + ops.sum(y_true)
        
        self.intersection.assign_add(intersection)
        self.union.assign_add(union)
    
    def result(self):
        return (2.0 * self.intersection + self.smooth) / (self.union + self.smooth)
    
    def reset_state(self):
        self.intersection.assign(0.0)
        self.union.assign(0.0)


print("Metrics defined")

In [ ]:
# =============================================================================
# CELL 9: CALLBACKS FOR model.fit()
# =============================================================================

class EpochUpdateCallback(keras.callbacks.Callback):
    """
    Callback to update epoch counter in loss function for progressive loss scheduling.
    
    This is necessary because CombinedLoss needs to know the current epoch
    to apply the correct loss weights (skeleton and clDice warmup).
    """
    
    def __init__(self, loss_fn):
        super().__init__()
        self.loss_fn = loss_fn
    
    def on_epoch_begin(self, epoch, logs=None):
        self.loss_fn.current_epoch = epoch
        
        # Print loss schedule status
        skel_scale = self.loss_fn._get_scale(
            epoch, 
            self.loss_fn.skeleton_start, 
            self.loss_fn.skeleton_warmup
        )
        cldice_scale = self.loss_fn._get_scale(
            epoch, 
            self.loss_fn.cldice_start, 
            self.loss_fn.cldice_warmup
        )
        
        if epoch % 50 == 0:  # Print every 50 epochs
            print(f"\n[Loss Schedule] Epoch {epoch}: "
                  f"skeleton_scale={skel_scale:.2f}, cldice_scale={cldice_scale:.2f}")


class GradientClipCallback(keras.callbacks.Callback):
    """
    Callback to apply gradient clipping during training.
    
    Note: Keras 3 supports clipnorm in optimizer, but this provides more control.
    """
    
    def __init__(self, clip_norm=12.0):
        super().__init__()
        self.clip_norm = clip_norm


print("Callbacks defined:")
print("  - EpochUpdateCallback: Updates loss scheduling based on current epoch")
print("  - GradientClipCallback: Gradient clipping support")

In [ ]:
# =============================================================================
# CELL 10: MODEL COMPILE & TRAINING SETUP (Using model.fit())
# =============================================================================

# Custom callback for periodic checkpoints (Keras 3 removed 'period' param)
class PeriodicCheckpoint(keras.callbacks.Callback):
    """Save model every N epochs."""
    def __init__(self, filepath, period=10, save_weights_only=True):
        super().__init__()
        self.filepath = filepath
        self.period = period
        self.save_weights_only = save_weights_only

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.period == 0:
            path = self.filepath.format(epoch=epoch + 1)
            if self.save_weights_only:
                self.model.save_weights(path)
            else:
                self.model.save(path)
            print(f"\nSaved checkpoint: {path}")


# Create loss function
criterion = CombinedLoss(
    dice_weight=cfg.DICE_WEIGHT,
    bce_weight=cfg.BCE_WEIGHT,
    skeleton_weight=cfg.SKELETON_WEIGHT,
    cldice_weight=cfg.CLDICE_WEIGHT,
    skeleton_start=cfg.SKELETON_START_EPOCH,
    skeleton_warmup=cfg.SKELETON_WARMUP_EPOCHS,
    cldice_start=cfg.CLDICE_START_EPOCH,
    cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
)

# Compile model with optimizer, loss, and metrics
# Using clipnorm in optimizer for gradient clipping (JAX compatible)
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=cfg.WEIGHT_DECAY,
    clipnorm=12.0,  # Gradient clipping
)

model.compile(
    optimizer=optimizer,
    loss=criterion,
    metrics=[DiceMetric(name='dice')],
)

print("Model compiled with:")
print(f"  Optimizer: AdamW (weight_decay={cfg.WEIGHT_DECAY}, clipnorm=12.0)")
print(f"  Loss: CombinedLoss (Dice + BCE + SkeletonRecall + clDice)")
print(f"  Metrics: Dice coefficient")

# Define callbacks
callbacks = [
    # Update epoch in loss function for progressive scheduling
    EpochUpdateCallback(criterion),

    # Save best model based on validation dice
    keras.callbacks.ModelCheckpoint(
        filepath=f"{cfg.CHECKPOINT_DIR}/best_model.weights.h5",
        monitor='val_dice',
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),

    # Save checkpoints every 10 epochs (custom callback - Keras 3 compatible)
    PeriodicCheckpoint(
        filepath=f"{cfg.CHECKPOINT_DIR}/checkpoint_epoch_{{epoch:04d}}.weights.h5",
        period=10,
        save_weights_only=True,
    ),

    # Log training history to CSV
    keras.callbacks.CSVLogger(
        f"{cfg.CHECKPOINT_DIR}/training_history.csv",
        append=True,
    ),

    # Reduce LR on plateau (backup scheduler)
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_dice',
        mode='max',
        factor=0.5,
        patience=50,
        min_lr=1e-7,
        verbose=1,
    ),
]

print(f"\nCallbacks configured:")
print(f"  - EpochUpdateCallback: Progressive loss scheduling")
print(f"  - ModelCheckpoint: Save best model")
print(f"  - PeriodicCheckpoint: Save every 10 epochs")
print(f"  - CSVLogger: Training history")
print(f"  - ReduceLROnPlateau: Backup LR scheduler")

In [ ]:
# =============================================================================
# CELL 11: RUN TRAINING WITH model.fit()
# =============================================================================

# Create tf.data datasets compatible with model.fit()
print("Creating tf.data datasets for model.fit()...")
train_ds = create_tf_dataset_for_fit(train_dataset, cfg.BATCH_SIZE, shuffle=True)
val_ds = create_tf_dataset_for_fit(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

# Calculate steps
train_steps = len(train_dataset) // cfg.BATCH_SIZE
val_steps = max(1, len(val_dataset) // cfg.BATCH_SIZE)

print(f"Training batches per epoch: {train_steps}")
print(f"Validation batches: {val_steps}")
print(f"Total epochs: {cfg.EPOCHS}")

print("\n" + "="*60)
print("STARTING TRAINING WITH model.fit()")
print("="*60)

# Run training using Keras model.fit()
# This automatically handles TF-to-JAX tensor conversion
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=cfg.EPOCHS,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    validation_freq=cfg.VALIDATE_EVERY,  # Validate every N epochs
    callbacks=callbacks,
    verbose=1,
)

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

# Save final model
model.save_weights(f"{cfg.CHECKPOINT_DIR}/final_model.weights.h5")
print(f"Final model saved to: {cfg.CHECKPOINT_DIR}/final_model.weights.h5")

In [ ]:
# =============================================================================
# CELL 12: EVALUATION WITH SLIDING WINDOW INFERENCE
# =============================================================================

def sliding_window_inference(
    model,
    volume: np.ndarray,
    roi_size: Tuple[int, int, int],
    overlap: float = 0.5,
    mode: str = 'gaussian',
) -> np.ndarray:
    """
    Sliding window inference for large volumes.
    
    Args:
        model: Trained model
        volume: Input volume (D, H, W) - will add channel dim
        roi_size: Patch size (tuple)
        overlap: Overlap fraction
        mode: 'gaussian' or 'constant' weighting
    
    Returns:
        Probability map (D, H, W)
    """
    # Normalize
    volume = (volume - volume.mean()) / (volume.std() + 1e-8)
    
    D, H, W = volume.shape
    pd, ph, pw = roi_size
    sd, sh, sw = int(pd * (1 - overlap)), int(ph * (1 - overlap)), int(pw * (1 - overlap))
    
    # Output arrays
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Create weight map
    if mode == 'gaussian':
        sigma = 0.125
        gauss_z = np.exp(-0.5 * ((np.arange(pd) - pd/2) / (pd * sigma)) ** 2)
        gauss_y = np.exp(-0.5 * ((np.arange(ph) - ph/2) / (ph * sigma)) ** 2)
        gauss_x = np.exp(-0.5 * ((np.arange(pw) - pw/2) / (pw * sigma)) ** 2)
        weight = gauss_z[:, None, None] * gauss_y[None, :, None] * gauss_x[None, None, :]
        weight = weight.astype(np.float32)
    else:
        weight = np.ones((pd, ph, pw), dtype=np.float32)
    
    # Generate positions
    z_pos = list(set(list(range(0, max(1, D-pd+1), sd)) + ([D-pd] if D > pd else [0])))
    y_pos = list(set(list(range(0, max(1, H-ph+1), sh)) + ([H-ph] if H > ph else [0])))
    x_pos = list(set(list(range(0, max(1, W-pw+1), sw)) + ([W-pw] if W > pw else [0])))
    
    total_patches = len(z_pos) * len(y_pos) * len(x_pos)
    print(f"Total patches: {total_patches}")
    
    patch_idx = 0
    for z in z_pos:
        for y in y_pos:
            for x in x_pos:
                # Extract patch
                patch = volume[z:z+pd, y:y+ph, x:x+pw]
                actual_shape = patch.shape
                
                # Pad if needed
                if patch.shape != (pd, ph, pw):
                    pad_d = pd - patch.shape[0]
                    pad_h = ph - patch.shape[1]
                    pad_w = pw - patch.shape[2]
                    patch = np.pad(patch, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect')
                
                # Add batch and channel dims: (D,H,W) -> (1,D,H,W,1)
                inp = patch[np.newaxis, ..., np.newaxis].astype(np.float32)
                
                # Inference
                out = model(inp, training=False)
                if isinstance(out, dict):
                    out = out['output']
                out = ops.sigmoid(out)
                out = np.array(out[0, ..., 0])  # Remove batch and channel dims
                
                # Crop to actual size
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                w = weight[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                
                # Accumulate
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * w
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += w
                
                patch_idx += 1
                if patch_idx % 10 == 0:
                    print(f"  Processed {patch_idx}/{total_patches} patches", end='\\r')
    
    print(f"  Processed {total_patches}/{total_patches} patches")
    
    return pred_sum / np.maximum(count, 1e-8)


def evaluate_on_volumes(model, data_root: str, volume_ids: List[str], roi_size, overlap=0.5):
    """
    Evaluate model on full volumes using sliding window inference.
    """
    data_root = Path(data_root)
    images_dir = data_root / "train_images"
    labels_dir = data_root / "train_labels"
    
    dice_scores = []
    
    for vol_id in volume_ids:
        print(f"\nEvaluating {vol_id}...")
        
        # Load volume and label
        volume = load_volume_tiff(images_dir / f"{vol_id}.tif").astype(np.float32)
        label = load_volume_tiff(labels_dir / f"{vol_id}.tif").astype(np.uint8)
        
        # Sliding window inference
        pred_prob = sliding_window_inference(model, volume, roi_size, overlap)
        
        # Threshold
        pred = (pred_prob > 0.5).astype(np.uint8)
        
        # Get ground truth (label==1 is foreground, label==2 is ignore)
        gt = (label == 1).astype(np.uint8)
        ignore = (label == 2)
        
        # Mask ignored regions
        pred[ignore] = 0
        gt[ignore] = 0
        
        # Dice
        intersection = (pred & gt).sum()
        union = pred.sum() + gt.sum()
        dice = (2 * intersection + 1e-5) / (union + 1e-5)
        dice_scores.append(dice)
        
        print(f"  Dice: {dice:.4f}")
        print(f"  Pred FG%: {100 * pred.mean():.2f}%")
        print(f"  GT FG%: {100 * gt.mean():.2f}%")
    
    mean_dice = np.mean(dice_scores)
    print(f"\\n{'='*40}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"{'='*40}")
    
    return mean_dice, dice_scores


print("Evaluation functions defined")

In [ ]:
# =============================================================================
# CELL 13: LOAD BEST MODEL AND EVALUATE
# =============================================================================

# Load best model weights
# model.load_weights(f"{cfg.CHECKPOINT_DIR}/best_model.weights.h5")

# Get some validation volume IDs for evaluation
# val_ids = val_dataset.volume_ids[:3]  # Evaluate on first 3 validation volumes

# Run evaluation with sliding window inference
# mean_dice, scores = evaluate_on_volumes(
#     model, 
#     cfg.DATA_ROOT, 
#     val_ids, 
#     cfg.PATCH_SIZE, 
#     cfg.VAL_OVERLAP
# )
# print(f"\\nFinal Mean Dice: {mean_dice:.4f}")

In [ ]:
# =============================================================================
# CELL 14: TRAINING STATUS CHECK
# =============================================================================

def check_training_status():
    """Check current training status."""
    import os
    
    checkpoint_dir = cfg.CHECKPOINT_DIR
    
    print("="*60)
    print("TRAINING STATUS")
    print("="*60)
    
    # Check for history
    history_path = f"{checkpoint_dir}/history.json"
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            history = json.load(f)
        
        print(f"\nTraining history: {len(history)} epochs")
        
        if len(history) > 0:
            # Find best
            best_epoch = max(history, key=lambda x: x.get('val_dice', 0))
            print(f"Best epoch: {best_epoch['epoch'] + 1}")
            print(f"Best Dice: {best_epoch.get('val_dice', 0):.4f}")
            
            # Recent epochs
            print("\nRecent epochs:")
            for h in history[-5:]:
                val_str = f", val_dice={h.get('val_dice', 0):.4f}" if h.get('val_dice', 0) > 0 else ""
                print(f"  Epoch {h['epoch']+1}: loss={h['train_loss']:.4f}{val_str}")
    else:
        print("\nNo training history found.")
    
    # Check for checkpoints
    checkpoints = glob.glob(f"{checkpoint_dir}/*.weights.h5")
    if checkpoints:
        print(f"\nCheckpoints found: {len(checkpoints)}")
        for ckpt in sorted(checkpoints)[-3:]:
            print(f"  {os.path.basename(ckpt)}")
    
    print("="*60)


check_training_status()